# port_congestion

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Load dataset
df = pd.read_csv("sample_data/port_congestion_synthetic_data.csv")
print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (200, 9)


,vessel_id,port_name,vessel_arrival_count,port_capacity,queue_length,weather_condition,historical_delay_hrs,arrival_delay_hrs,congestion_level
0,VSL_001,Port of Shanghai,6,15,12,Moderate,4.77,14.57,High
1,VSL_002,Port of Nhava Sheva,10,10,2,Severe,11.63,17.33,High
2,VSL_003,Port of Los Angeles,26,25,1,Moderate,10.39,9.72,Medium
3,VSL_004,Port of Nhava Sheva,15,10,2,Moderate,9.80,11.64,Medium
4,VSL_005,Port of Nhava Sheva,20,25,6,Moderate,3.09,9.46,Medium


In [ ]:
# One-hot encode categorical features
df_model = pd.get_dummies(df, columns=["weather_condition", "port_name"], drop_first=True)

X = df_model.drop(columns=["vessel_id", "arrival_delay_hrs", "congestion_level"])
y = df_model["arrival_delay_hrs"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train Random Forest Regressor
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

# Evaluate model performance
preds = rf.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, preds))
r2 = r2_score(y_test, preds)

print(f"Model Training Complete!")
print(f"RMSE: {rmse:.2f} hours")
print(f"R2 Score: {r2:.2f}")

Model Training Complete!
RMSE: 2.24 hours
R2 Score: 0.75


In [ ]:
def predict_port_congestion(vessel_arrival_count, port_capacity, queue_length, historical_delay_hrs, weather_condition, port_name):
    input_data = {
        'vessel_arrival_count': [vessel_arrival_count],
        'port_capacity': [port_capacity],
        'queue_length': [queue_length],
        'historical_delay_hrs': [historical_delay_hrs]
    }

    for col in X.columns:
        if col.startswith('weather_condition_') or col.startswith('port_name_'):
            input_data[col] = [1 if col == f"weather_condition_{weather_condition}" or col == f"port_name_{port_name}" else 0]

    input_df = pd.DataFrame(input_data)[X.columns]

    # Predict delay
    pred_delay = rf.predict(input_df)[0]

    # Confidence Score using tree variance
    tree_preds = [tree.predict(input_df)[0] for tree in rf.estimators_]
    std_dev = np.std(tree_preds)
    confidence_score = max(0, min(100, round(100 - (std_dev * 10), 2)))

    # Explanation factors
    reasons = []
    if vessel_arrival_count > port_capacity:
        reasons.append(f"High vessel traffic ({vessel_arrival_count}) exceeding port capacity ({port_capacity})")
    if queue_length > 5:
        reasons.append(f"Long operational queue length of {queue_length} vessels")
    if weather_condition in ["Bad", "Severe"]:
        reasons.append(f"Adverse weather condition ({weather_condition}) affecting port operations")
    if historical_delay_hrs > 5:
        reasons.append(f"High historical delay baseline ({historical_delay_hrs} hrs)")

    if not reasons:
        reasons.append("Normal operational parameters with minimal congestion risk.")

    return {
        "predicted_delay_hrs": round(pred_delay, 2),
        "confidence_score": confidence_score,
        "key_factors": reasons
    }

# Test with a high congestion scenario
test_result = predict_port_congestion(
    vessel_arrival_count=30,
    port_capacity=15,
    queue_length=10,
    historical_delay_hrs=7.0,
    weather_condition="Bad",
    port_name="Port of Singapore"
)

import pprint
pprint.pprint(test_result)

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but DecisionTreeRegressor was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but DecisionTreeRegressor was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but DecisionTreeRegressor was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but DecisionTreeRegressor was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but DecisionTreeRegressor was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2732: UserWa

{'confidence_score': np.float64(60.43),
 'key_factors': ['High vessel traffic (30) exceeding port capacity (15)',
                 'Long operational queue length of 10 vessels',
                 'Adverse weather condition (Bad) affecting port operations',
                 'High historical delay baseline (7.0 hrs)'],
 'predicted_delay_hrs': np.float64(20.96)}


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but DecisionTreeRegressor was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but DecisionTreeRegressor was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but DecisionTreeRegressor was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but DecisionTreeRegressor was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but DecisionTreeRegressor was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2732: UserWa